In [1]:
%use kandy

@file:DependsOn("org.oremif:kstats-core-jvm:0.3.0")
@file:DependsOn("org.oremif:kstats-distributions-jvm:0.3.0")
@file:DependsOn("org.oremif:kstats-hypothesis-jvm:0.3.0")
@file:DependsOn("org.oremif:kstats-correlation-jvm:0.3.0")

In [2]:
import org.oremif.kstats.descriptive.*
import org.oremif.kstats.hypothesis.*
import org.oremif.kstats.correlation.*

# Building a Statistics Pipeline

As analysis grows beyond one-off calls, wrapping kstats functions into typed pipeline functions keeps the workflow repeatable and the results structured.

## Result Types

In [3]:
data class AssumptionCheck(
    val normalityPValue: Double,
    val isNormal: Boolean,
    val varianceEqualityPValue: Double,
    val isVarianceEqual: Boolean
)

data class GroupComparison(
    val testName: String,
    val pValue: Double,
    val isSignificant: Boolean,
    val confidenceInterval: Pair<Double, Double>?
)

data class AnalysisReport(
    val controlSummary: DescriptiveStatistics,
    val treatmentSummary: DescriptiveStatistics,
    val assumptions: AssumptionCheck,
    val comparison: GroupComparison
)

## Check Assumptions

In [4]:
fun checkAssumptions(
    control: DoubleArray,
    treatment: DoubleArray,
    alpha: Double = 0.05
): AssumptionCheck {
    val controlNormality = shapiroWilkTest(control)
    val treatmentNormality = shapiroWilkTest(treatment)
    val normality = minOf(controlNormality.pValue, treatmentNormality.pValue)

    val variance = leveneTest(control, treatment)

    return AssumptionCheck(
        normalityPValue = normality,
        isNormal = normality >= alpha,
        varianceEqualityPValue = variance.pValue,
        isVarianceEqual = variance.pValue >= alpha
    )
}

## Compare Groups

In [5]:
fun compareGroups(
    control: DoubleArray,
    treatment: DoubleArray,
    assumptions: AssumptionCheck,
    alpha: Double = 0.05
): GroupComparison {
    val result = if (assumptions.isNormal) {
        tTest(control, treatment, equalVariances = assumptions.isVarianceEqual)
    } else {
        mannWhitneyUTest(control, treatment)
    }

    return GroupComparison(
        testName = result.testName,
        pValue = result.pValue,
        isSignificant = result.isSignificant(alpha),
        confidenceInterval = result.confidenceInterval
    )
}

## Full Report

In [6]:
fun analyze(
    control: DoubleArray,
    treatment: DoubleArray,
    alpha: Double = 0.05
): AnalysisReport {
    val assumptions = checkAssumptions(control, treatment, alpha)
    val comparison = compareGroups(control, treatment, assumptions, alpha)

    return AnalysisReport(
        controlSummary = control.describe(),
        treatmentSummary = treatment.describe(),
        assumptions = assumptions,
        comparison = comparison
    )
}

## Usage

In [7]:
val pageLoadControl = doubleArrayOf(
    1.23, 1.45, 1.31, 1.52, 1.38, 1.41, 1.29, 1.47, 1.35, 1.44,
    1.33, 1.50, 1.27, 1.42, 1.36, 1.48, 1.30, 1.46, 1.39, 1.43
)
val pageLoadTreatment = doubleArrayOf(
    1.10, 1.25, 1.18, 1.32, 1.15, 1.22, 1.12, 1.28, 1.19, 1.26,
    1.14, 1.30, 1.11, 1.24, 1.17, 1.29, 1.13, 1.27, 1.20, 1.23
)

val report = analyze(pageLoadControl, pageLoadTreatment)

println("=== Analysis Report ===")
println("Control mean:     ${String.format("%.3f", report.controlSummary.mean)} s")
println("Treatment mean:   ${String.format("%.3f", report.treatmentSummary.mean)} s")
println("Normal:           ${report.assumptions.isNormal} (p=${String.format("%.4f", report.assumptions.normalityPValue)})")
println("Equal variance:   ${report.assumptions.isVarianceEqual} (p=${String.format("%.4f", report.assumptions.varianceEqualityPValue)})")
println("Test used:        ${report.comparison.testName}")
println("p-value:          ${String.format("%.6f", report.comparison.pValue)}")
println("Significant:      ${report.comparison.isSignificant}")
println("95% CI:           ${report.comparison.confidenceInterval?.let { "[${String.format("%.3f", it.first)}, ${String.format("%.3f", it.second)}]" } ?: "N/A"}")

=== Analysis Report ===
Control mean:     1,390 s
Treatment mean:   1,208 s
Normal:           true (p=0,4640)
Equal variance:   true (p=0,4181)
Test used:        Two-Sample t-Test (Equal Variances)
p-value:          0,000000
Significant:      true
95% CI:           [0,134, 0,230]


In [8]:
plot {
    densityPlot(pageLoadControl.toList()) {
        fillColor = Color.hex("#4dabf7")
        alpha = 0.4
    }
    densityPlot(pageLoadTreatment.toList()) {
        fillColor = Color.hex("#ff8787")
        alpha = 0.4
    }
    layout { title = "Page Load Times: Control (blue) vs Treatment (red)"; size = 800 to 400 }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="tEbyHo"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("tEbyHo");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Page Load Times: Control (blue) vs Treatment (red)"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":800.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"density"
},
"stat":"identity",
"data":{
"density":[1.1654787569015763,1.1790464820864806,1.1926791478458696,1.206375988057523,1.2201362155781998,1.2339590214941032,1.247843574369904,1.2617890194983037,1.2757944781519819,1.2898590468398101,1.3039817965694256,1.3181617721179937,1.3323979913133033,1.3466894443271635,1.3610350929831063,1.3754338700805586,1.3898846787373917,1.4043863917530754,1.4189378509943036,1.4335378668053,1.448185217444759,1.4628786485513807,1.4776168726401457,1.4923985686311052,1.5072223814127654,1.5220869214419153,1.5369907643816665,1.5519324507796903,1.5669104857882092,1.5819233389276017,1.5969694438952051,1.612047198420838,1.6271549641706948,1.6422910667008914,1.6574537954621555,1.6726414038569104,1.6878521093498704,1.7030840936334424,1.7183355028487555,1.7336044478634094,1.7488890046067318,1.764187214463202,1.779497084724844,1.794816589102919,1.8101436682995566,1.8254762306394323,1.8408121527618717,1.8561492803733919,1.8714854290606264,1.8868183851636402,1.9021459067091993,1.9174657244038045,1.9327755426859272,1.9480730408368099,1.9633558741492574,1.9786216751534185,1.9938680548987702,2.009092604291177,2.024292895483803,2.0394664833207505,2.054610906831844,2.0697236907772174,2.0848023472400414,2.0998443772656104,2.1148472725451475,2.1298085171422283,2.1447255892599646,2.1595959630468142,2.174417110438752,2.1891865030356947,2.2039016140096583,2.2185599200423494,2.233158903289667,2.2476960533704573,2.2621688693770645,2.2765748619047903,2.2909115550977273,2.305176488707979,2.319367220165588,2.3334813266562464,2.347516407203809,2.361470084754848,2.3753400082621043,2.3891238547640175,2.4028193314572848,2.41642417775944,2.429936167358553,2.4433531102469663,2.4566728547361945,2.4698932894500025,2.483012345292683,2.496027997389762,2.508938266998125,2.521741223382861,2.534434985657999,2.547017724588366,2.559487664350013,2.5718430842464564,2.5840823203783327,2.596203767263873,2.6082058794078775,2.6200871728168718,2.631846226458122,2.643481683660524,2.654992253455131,2.666376711853519,2.677633903062058,2.688762740630319,2.699762208532041,2.710631362177046,2.7213693293527204,2.731975311093762,2.74244858247893,2.7527884933538145,2.7629944689785804,2.7730660105999188,2.7830026959464655,2.792804179647068,2.8024701935715095,2.8120005470932594,2.8213951272741156,2.830653898970612,2.8397769048621857,2.84876426540136,2.8576161786860963,2.8663329202548313,2.8749148428046607,2.8833623758332685,2.8916760252054825,2.8998563726451647,2.907904075153567,2.9158198643551296,2.9236045457719673,2.931258998028366,2.9387841719866277,2.94618108981580

In [9]:
val ciLow = report.comparison.confidenceInterval?.first ?: 0.0
val ciHigh = report.comparison.confidenceInterval?.second ?: 0.0
val meanDiff = report.controlSummary.mean - report.treatmentSummary.mean

plot {
    bars {
        x(listOf("Control", "Treatment")); y(listOf(report.controlSummary.mean, report.treatmentSummary.mean))
        fillColor = Color.hex("#4dabf7")
        alpha = 0.7
    }
    layout { title = "Mean Page Load Times"; size = 500 to 400 }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="rZDyEU"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 500.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("rZDyEU");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Mean Page Load Times"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":500.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Control","Treatment"],
"y":[1.3895000000000002,1.2075000000000002]
},
"sampling":"none",
"alpha":0.7,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"fill":"#4dabf7",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"5"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Control 
 
 
 
 
 
 
 
 
 Treatment 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 1.2 
 
 
 
 
 
 
 1.4 
 
 
 
 
 
 
 
 
 Mean Page Load Times 
 
 
 
 
 y 
 
 
 
 
 x

In [10]:
plot {
    points {
        x(listOf(meanDiff)); y(listOf("Difference"))
        size = 8.0
        color = Color.hex("#1971c2")
    }
    line {
        x(listOf(ciLow, ciHigh)); y(listOf("Difference", "Difference"))
        color = Color.hex("#1971c2")
        width = 3.0
    }
    line {
        x(listOf(0.0, 0.0)); y(listOf("", "Difference"))
        color = Color.hex("#e03131")
        width = 1.5
    }
    layout { title = "Mean Difference with 95% CI"; size = 700 to 250 }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="FwB7Sh"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 250.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("FwB7Sh");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Mean Difference with 95% CI"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":250.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"discrete":true
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"discrete":true
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.18199999999999994],
"y":["Difference"]
},
"size":8.0,
"color":"#1971c2",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"str",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.13366044256539414,0.23033955743460574],
"y":["Difference","Difference"]
},
"color":"#1971c2",
"size":3.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"str",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.0,0.0],
"y":["","Difference"]
},
"color":"#e03131",
"size":1.5,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"str",
"column":"y"
}]
}
}],
"spec_id":"8"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 0.05 
 
 
 
 
 
 
 
 
 0.1 
 
 
 
 
 
 
 
 
 0.15 
 
 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 
 
 
 
 Difference 
 
 
 
 
 
 
 
 
 
 
 
 
 Mean Difference with 95% CI 
 
 
 
 
 y 
 
 
 
 
 x

## Extending the Pipeline

In [11]:
data class ExtendedReport(
    val base: AnalysisReport,
    val correlationCoefficient: Double,
    val correlationPValue: Double,
    val regressionSlope: Double,
    val regressionRSquared: Double
)

fun analyzeWithCorrelation(
    control: DoubleArray,
    treatment: DoubleArray,
    metricX: DoubleArray,
    metricY: DoubleArray
): ExtendedReport {
    val base = analyze(control, treatment)
    val correlation = pearsonCorrelation(metricX, metricY)
    val regression = simpleLinearRegression(metricX, metricY)

    return ExtendedReport(
        base = base,
        correlationCoefficient = correlation.coefficient,
        correlationPValue = correlation.pValue,
        regressionSlope = regression.slope,
        regressionRSquared = regression.rSquared
    )
}

In [12]:
// Combine both groups for correlation analysis
val allPageLoads = pageLoadControl + pageLoadTreatment
val bounceRates = doubleArrayOf(
    0.42, 0.38, 0.40, 0.35, 0.39, 0.37, 0.41, 0.34, 0.38, 0.36,
    0.40, 0.33, 0.43, 0.37, 0.39, 0.34, 0.41, 0.36, 0.38, 0.37,
    0.30, 0.25, 0.28, 0.22, 0.27, 0.24, 0.29, 0.21, 0.26, 0.23,
    0.28, 0.20, 0.30, 0.24, 0.27, 0.21, 0.29, 0.23, 0.25, 0.24
)

val extended = analyzeWithCorrelation(pageLoadControl, pageLoadTreatment, allPageLoads, bounceRates)

println("Correlation (load time \u2194 bounce): ${String.format("%.3f", extended.correlationCoefficient)} (p=${String.format("%.4f", extended.correlationPValue)})")
println("Regression slope: ${String.format("%.3f", extended.regressionSlope)}")
println("R\u00b2: ${String.format("%.3f", extended.regressionRSquared)}")

Correlation (load time ↔ bounce): 0,461 (p=0,0027)
Regression slope: 0,273
R²: 0,213


In [13]:
val reg = simpleLinearRegression(allPageLoads, bounceRates)
val xMin = allPageLoads.min()
val xMax = allPageLoads.max()

plot {
    points {
        x(allPageLoads.toList()); y(bounceRates.toList())
        color = Color.hex("#7048e8")
        size = 5.0
    }
    line {
        x(listOf(xMin, xMax)); y(listOf(reg.predict(xMin), reg.predict(xMax)))
        color = Color.hex("#e03131")
        width = 2.0
    }
    layout { title = "Page Load Time vs Bounce Rate (R\u00b2=${String.format("%.3f", reg.rSquared)})"; size = 700 to 500 }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="rH7Q24"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 500.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("rH7Q24");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Page Load Time vs Bounce Rate (R²=0,213)"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[1.23,1.45,1.31,1.52,1.38,1.41,1.29,1.47,1.35,1.44,1.33,1.5,1.27,1.42,1.36,1.48,1.3,1.46,1.39,1.43,1.1,1.25,1.18,1.32,1.15,1.22,1.12,1.28,1.19,1.26,1.14,1.3,1.11,1.24,1.17,1.29,1.13,1.27,1.2,1.23],
"y":[0.42,0.38,0.4,0.35,0.39,0.37,0.41,0.34,0.38,0.36,0.4,0.33,0.43,0.37,0.39,0.34,0.41,0.36,0.38,0.37,0.3,0.25,0.28,0.22,0.27,0.24,0.29,0.21,0.26,0.23,0.28,0.2,0.3,0.24,0.27,0.21,0.29,0.23,0.25,0.24]
},
"color":"#7048e8",
"size":5.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[1.1,1.52],
"y":[0.2618165392126444,0.3764616451607016]
},
"color":"#e03131",
"size":2.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"11"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1.1 
 
 
 
 
 
 
 
 
 1.15 
 
 
 
 
 
 
 
 
 1.2 
 
 
 
 
 
 
 
 
 1.25 
 
 
 
 
 
 
 
 
 1.3 
 
 
 
 
 
 
 
 
 1.35 
 
 
 
 
 
 
 
 
 1.4 
 
 
 
 
 
 
 
 
 1.45 
 
 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.22 
 
 
 
 
 
 
 0.24 
 
 
 
 
 
 
 0.26 
 
 
 
 
 
 
 0.28 
 
 
 
 
 
 
 0.3 
 
 
 
 
 
 
 0.32 
 
 
 
 
 
 
 0.34 
 
 
 
 
 
 
 0.36 
 
 
 
 
 
 
 0.38 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.42 
 
 
 
 
 
 
 0.44 
 
 
 
 
 
 
 
 
 Page Load Time vs Bounce Rate (R²=0,213) 
 
 
 
 
 y 
 
 
 
 
 x

## Running on Multiple Datasets

In [14]:
// Experiment 1: Page load optimization (same as above)
val exp1 = report

// Experiment 2: Checkout flow
val checkoutControl = doubleArrayOf(4.2, 5.1, 4.8, 5.3, 4.5, 5.0, 4.7, 5.2, 4.4, 4.9)
val checkoutTreatment = doubleArrayOf(3.8, 4.3, 4.0, 4.5, 3.7, 4.2, 3.9, 4.4, 3.6, 4.1)
val exp2 = analyze(checkoutControl, checkoutTreatment)

// Experiment 3: Search latency (no real effect)
val searchControl = doubleArrayOf(0.52, 0.48, 0.55, 0.50, 0.53, 0.49, 0.51, 0.54, 0.47, 0.52)
val searchTreatment = doubleArrayOf(0.51, 0.49, 0.53, 0.50, 0.52, 0.48, 0.50, 0.53, 0.48, 0.51)
val exp3 = analyze(searchControl, searchTreatment)

println("Experiment 1 (Page Load):  p=${String.format("%.4f", exp1.comparison.pValue)}, sig=${exp1.comparison.isSignificant}")
println("Experiment 2 (Checkout):   p=${String.format("%.4f", exp2.comparison.pValue)}, sig=${exp2.comparison.isSignificant}")
println("Experiment 3 (Search):     p=${String.format("%.4f", exp3.comparison.pValue)}, sig=${exp3.comparison.isSignificant}")

Experiment 1 (Page Load):  p=0,0000, sig=true
Experiment 2 (Checkout):   p=0,0001, sig=true
Experiment 3 (Search):     p=0,5590, sig=false


In [15]:
val expNames = listOf("Page Load", "Checkout", "Search")
val expPvalues = listOf(exp1.comparison.pValue, exp2.comparison.pValue, exp3.comparison.pValue)

plot {
    bars {
        x(expNames); y(expPvalues)
        fillColor = Color.hex("#ffd43b")
        alpha = 0.8
    }
    line {
        x(listOf(expNames.first(), expNames.last())); y(listOf(0.05, 0.05))
        color = Color.hex("#e03131")
        width = 2.0
    }
    layout { title = "Experiment p-values (red = \u03b1=0.05)"; size = 700 to 400 }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="t4ku7W"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("t4ku7W");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Experiment p-values (red = α=0.05)"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Page Load","Checkout","Search"],
"y":[3.6002268378467307E-9,7.386624959384047E-5,0.5589968276322217]
},
"sampling":"none",
"alpha":0.8,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"fill":"#ffd43b",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Page Load","Search"],
"y":[0.05,0.05]
},
"color":"#e03131",
"size":2.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"14"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Page Load 
 
 
 
 
 
 
 
 
 Checkout 
 
 
 
 
 
 
 
 
 Search 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.1 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.3 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 
 
 Experiment p-values (red = α=0.05) 
 
 
 
 
 y 
 
 
 
 
 x

## Module Responsibilities

| Pipeline stage | Module | Key functions |
| --- | --- | --- |
| Normalize, rank, bin, resample | `kstats-sampling` | `zScore()`, `rank()`, `bin()`, `bootstrapSample()` |
| Summarize | `kstats-core` | `describe()`, `mean()`, `quantile()` |
| Check assumptions, compare groups | `kstats-hypothesis` | `shapiroWilkTest()`, `leveneTest()`, `tTest()` |
| Model relationships | `kstats-correlation` | `pearsonCorrelation()`, `simpleLinearRegression()` |
| Estimate probabilities | `kstats-distributions` | `NormalDistribution()`, `cdf()`, `quantile()` |

## Summary

A typed pipeline wraps kstats calls into composable functions. By separating assumption checking from test selection, the pipeline automatically adapts to data characteristics.